# IRRA + LGSP-inspired prompts — ICFG-PEDES

1. Bật **GPU T4**, Internet; chỉ GPU 0 được dùng.
2. Add Input: ICFG-PEDES, output/dataset chứa IRRA `best.pth` (kèm `configs.yaml` nếu có),
   và `dist/irra_lgsp_source.zip` tạo từ working tree này. Không cần push Git để dùng ZIP.
3. Chạy từ trên xuống. Mặc định: đánh giá checkpoint gốc, smoke test, **E2 frequency 5 epoch**,
   **E0 baseline thêm cùng 5 epoch**, rồi test chính xác theo chunk.
4. Cố định thiết lập trước khi test cuối. Chạy lại screening trên validation, không chọn siêu tham số từ test.

Mã nguồn đã tích hợp các bản vá đánh giá trong notebook baseline; không có cell sửa source.
Đây là thử nghiệm chuyển cơ chế LGSP sang retrieval, không phải tái tạo toàn bộ LGSP/FSCIL.
Không có kết quả benchmark được điền sẵn.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # before importing torch

EXPERIMENT = 'frequency'  # baseline | frequency | local | both
INIT_CHECKPOINT = None  # explicit /kaggle/input/.../best.pth when ambiguous
BASE_CONFIG = None      # default: configs.yaml beside INIT_CHECKPOINT, if present
ANNOTATION_SOURCE = None
IMAGES_SOURCE = None
SOURCE_ZIP = None       # default: find exactly one irra_lgsp_source.zip
ALLOW_GIT_CLONE = False # enable only after you have pushed the research branch
REPO_URL = 'https://github.com/thanhphu25/IRRA.git'
RESEARCH_REF = 'feat/lgsp-tbpr'

EPOCHS = 5
BATCH_SIZE = 64
TEST_BATCH_SIZE = 128
NUM_WORKERS = 2
FREEZE_BACKBONE = False
SEED = 42
BACKBONE_LR = 1e-5
PROMPT_LR = 1e-3
FREQUENCY_RINGS = 8
FREQUENCY_VARIANT = 'real'
FREQUENCY_ALPHA = 1e-3
LOCAL_POOL_SIZE = 4
LOCAL_HIDDEN_DIM = 8
LOCAL_CHECKPOINT = False
VALIDATION_SPLIT = 'auto'  # val if supplied, otherwise train identity holdout
HOLDOUT_SEED = 42
HOLDOUT_FRACTION = 0.1
RUN_BASELINE_EVALUATION = True
RUN_MATCHED_CONTROL = True
RUN_UNIT_TESTS = True
RUN_FINAL_TEST = True     # one final evaluation after settings are fixed
OUTPUT_ROOT = '/kaggle/working/IRRA_LGSP_runs'
DATA_ROOT = '/kaggle/working/data'

assert EXPERIMENT in ('baseline', 'frequency', 'local', 'both')
assert not (FREEZE_BACKBONE and (EXPERIMENT == 'baseline' or RUN_MATCHED_CONTROL)), (
    'Frozen-backbone is a prompt-only diagnostic: select a prompt mode and set RUN_MATCHED_CONTROL=False.'
)

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import hashlib
import zipfile
import uuid

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
source_candidates = [Path(SOURCE_ZIP)] if SOURCE_ZIP else list(INPUT_ROOT.rglob('irra_lgsp_source.zip'))
if len(source_candidates) > 1:
    raise ValueError(f'Set SOURCE_ZIP; found {source_candidates}')
if source_candidates:
    destination = WORK_ROOT / ('irra_lgsp_source_' + uuid.uuid4().hex[:10])
    destination.mkdir()
    with zipfile.ZipFile(source_candidates[0]) as z:
        for name in z.namelist():
            target = (destination / name).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError(f'Unsafe ZIP member: {name}')
        z.extractall(destination)
    REPO = destination / 'IRRA'
elif (WORK_ROOT / 'IRRA/model/lgsp/spatial_fusion.py').is_file():
    REPO = WORK_ROOT / 'IRRA'
elif ALLOW_GIT_CLONE:
    REPO = WORK_ROOT / ('IRRA_' + uuid.uuid4().hex[:10])
    subprocess.run(['git', 'clone', '--branch', RESEARCH_REF, '--single-branch', REPO_URL, str(REPO)], check=True)
else:
    raise FileNotFoundError('Add irra_lgsp_source.zip as Kaggle Input, or use an existing research checkout.')

assert (REPO / 'model/lgsp/frequency_prompt.py').is_file()
manifest_path = REPO / 'bundle_manifest.json'
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
    for relative, expected in manifest['files'].items():
        assert hashlib.sha256((REPO / relative).read_bytes()).hexdigest() == expected, relative
    print('Source branch/base commit:', manifest['branch'], manifest['base_commit'])
    source_provenance = manifest
else:
    source_provenance = dict(commit=subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source:', REPO)

def run(command):
    import shlex
    print(shlex.join([str(s) for s in command]), flush=True)
    subprocess.run([str(s) for s in command], cwd=REPO, check=True)

run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'experiments/lgsp_tbpr/requirements.txt'])
import torch
import torchvision
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__, 'CUDA:', torch.version.cuda)
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from experiments.lgsp_tbpr.prepare_kaggle import prepare_icfg, find_checkpoint

dataset_info = prepare_icfg(INPUT_ROOT, DATA_ROOT, ANNOTATION_SOURCE, IMAGES_SOURCE)
INIT_CHECKPOINT = find_checkpoint(INPUT_ROOT, INIT_CHECKPOINT)
if BASE_CONFIG is None and (INIT_CHECKPOINT.parent / 'configs.yaml').is_file():
    BASE_CONFIG = INIT_CHECKPOINT.parent / 'configs.yaml'
if VALIDATION_SPLIT == 'auto':
    VALIDATION_SPLIT = ('val' if any(s not in ('train', 'test') and n > 0
                                   for s, n in dataset_info['splits'].items()) else 'holdout')
print('Checkpoint:', INIT_CHECKPOINT)
print('Original config:', BASE_CONFIG or 'unavailable; standard notebook baseline settings are assumed')
print('Validation:', VALIDATION_SPLIT)
if VALIDATION_SPLIT == 'holdout':
    print('Train-only screening holdout; these identities were already seen by the old pretrained IRRA.')
    print('An unbiased holdout comparison requires retraining E0/E2 from CLIP while excluding the same identities.')

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
environment = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(Path(OUTPUT_ROOT) / 'environment.txt').write_text(environment)
(Path(OUTPUT_ROOT) / 'protocol.json').write_text(json.dumps(dict(
    dataset=dataset_info, checkpoint=str(INIT_CHECKPOINT), seed=SEED, batch_size=BATCH_SIZE,
    epochs=EPOCHS, validation=VALIDATION_SPLIT, holdout_seed=HOLDOUT_SEED,
    holdout_fraction=HOLDOUT_FRACTION, torch=str(torch.__version__), cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0), source=source_provenance), indent=2))

CLIP ViT-B/16 được tải tự động qua bộ nạp IRRA trong smoke test đầu tiên.
Nếu Internet bị tắt, cần gắn sẵn CLIP pretrained và cache theo README.
`train_features.pt`/`test_features.pt` không được dùng để huấn luyện prompt.

Giữ batch 64; nếu OOM hãy giảm **cả E0 và E2** về cùng batch size.
SDM dùng quan hệ trong batch, nên gradient accumulation không thay thế batch 64 tương đương.

In [ ]:
def common_args():
    args = ['--config_file', str(BASE_CONFIG)] if BASE_CONFIG else []
    args += ['--pretrain_choice', 'ViT-B/16', '--dataset_name', 'ICFG-PEDES',
        '--root_dir', DATA_ROOT, '--img_aug', '--MLM', '--loss_names', 'sdm+mlm+id',
        '--batch_size', str(BATCH_SIZE), '--test_batch_size', str(TEST_BATCH_SIZE),
        '--num_workers', str(NUM_WORKERS), '--num_epoch', str(EPOCHS), '--warmup_epochs', '1',
        '--lr', str(BACKBONE_LR), '--lgsp_lr', str(PROMPT_LR), '--seed', str(SEED), '--deterministic',
        '--val_dataset', VALIDATION_SPLIT, '--holdout_seed', str(HOLDOUT_SEED),
        '--holdout_fraction', str(HOLDOUT_FRACTION), '--eval_chunk_size', '128',
        '--frequency_num_rings', str(FREQUENCY_RINGS), '--frequency_variant', FREQUENCY_VARIANT,
        '--frequency_alpha_init', str(FREQUENCY_ALPHA), '--local_pool_size', str(LOCAL_POOL_SIZE),
        '--local_hidden_dim', str(LOCAL_HIDDEN_DIM), '--lgsp_diagnostics']
    if LOCAL_CHECKPOINT:
        args += ['--local_checkpoint']
    return args

if RUN_UNIT_TESTS:
    run([sys.executable, '-m', 'pytest', 'tests', '-q'])

In [ ]:
# Measure the user's original checkpoint independently of continued fine-tuning.
import yaml
from utils.options import get_args

BASELINE_EVAL_DIR = Path(OUTPUT_ROOT) / ('initial_baseline_' + uuid.uuid4().hex[:8])
if RUN_BASELINE_EVALUATION:
    BASELINE_EVAL_DIR.mkdir()
    baseline_config = get_args(common_args() + ['--lgsp_mode', 'baseline', '--output_dir', str(BASELINE_EVAL_DIR)])
    baseline_config.freeze_backbone = False
    baseline_config.training = False
    config_path = BASELINE_EVAL_DIR / 'configs.yaml'
    config_path.write_text(yaml.safe_dump(vars(baseline_config)))
    run([sys.executable, 'test.py', '--config_file', config_path, '--checkpoint', INIT_CHECKPOINT])
    print('Measured original baseline:', (BASELINE_EVAL_DIR / 'metrics.json').read_text())

In [ ]:
# A separate process releases the smoke model and optimizer before actual training.
smoke_args = common_args() + ['--lgsp_mode', EXPERIMENT, '--init_checkpoint', str(INIT_CHECKPOINT),
    '--output_dir', str(Path(OUTPUT_ROOT) / ('smoke_' + EXPERIMENT + '_' + uuid.uuid4().hex[:8]))]
if FREEZE_BACKBONE:
    smoke_args += ['--freeze_backbone']
run([sys.executable, '-m', 'experiments.lgsp_tbpr.smoke_test'] + smoke_args)

Lần huấn luyện đầu tiên mặc định là **E2**. Sau đó chạy E0 với cùng checkpoint,
số epoch, augmentation, split, seed, batch và learning rate của các lớp IRRA cũ.
E1/E3 được chọn bằng `EXPERIMENT='local'/'both'` sau khi đã kiểm tra E2.
Frozen-backbone là chẩn đoán riêng; không so trực tiếp với E0 full fine-tuning.

In [ ]:
IDS = dict(baseline='E0', local='E1', frequency='E2', both='E3')
RUNS = {}
def train_experiment(mode):
    label = IDS[mode] + '_' + mode
    root = Path(OUTPUT_ROOT) / label
    before = set(root.rglob('configs.yaml')) if root.exists() else set()
    args = common_args() + ['--name', label, '--lgsp_mode', mode,
        '--init_checkpoint', str(INIT_CHECKPOINT), '--output_dir', str(root)]
    if FREEZE_BACKBONE:
        args += ['--freeze_backbone']
    run([sys.executable, 'train.py'] + args)
    configs = set(root.rglob('configs.yaml')) - before
    assert len(configs) == 1, configs
    config = configs.pop()
    assert (config.parent / 'best.pth').is_file()
    import shutil
    for name in ('protocol.json', 'environment.txt'):
        shutil.copy2(Path(OUTPUT_ROOT) / name, config.parent / name)
    RUNS[label] = config
    print(label, 'config:', config, 'best:', config.parent / 'best.pth')

train_experiment(EXPERIMENT)
if RUN_MATCHED_CONTROL and EXPERIMENT != 'baseline':
    train_experiment('baseline')

In [ ]:
import csv
summary = []
for label, config in RUNS.items():
    if RUN_FINAL_TEST:
        run([sys.executable, 'test.py', '--config_file', config])
        metrics = json.loads((config.parent / 'metrics.json').read_text())['t2i']
        summary.append(dict(experiment=label, seed=SEED, batch_size=BATCH_SIZE,
            epochs=EPOCHS, validation=VALIDATION_SPLIT, **metrics, output_dir=str(config.parent)))
    print('Artifacts:', config.parent)
    for name in ['configs.yaml', 'train_log.txt', 'best.pth', 'last.pth', 'metrics.json',
                 'training_stats.jsonl', 'validation_metrics.jsonl']:
        print(' ', name, (config.parent / name).exists())
if summary:
    destination = Path(OUTPUT_ROOT) / ('summary_' + uuid.uuid4().hex[:8] + '.csv')
    with destination.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(summary[0]))
        writer.writeheader()
        writer.writerows(summary)
    print('CSV:', destination)
    for row in summary:
        print(row)
print('Save Version → output files under:', OUTPUT_ROOT)

Đầu ra `metrics.json` chứa R1/R5/R10/mAP/mINP theo phần trăm; `summary_*.csv` tổng hợp
kết quả **thực tế** sau khi chạy. `best.pth` chứa toàn bộ IRRA và prompt; `last.pth` hỗ trợ resume.
Resume: `python train.py --resume --resume_ckpt_file <run>/last.pth` tự đọc `<run>/configs.yaml`.
Với run lấy từ Kaggle Input (chỉ đọc), sao chép cả thư mục run vào Working trước khi resume.
Không thay tổng số epoch khi resume cùng scheduler; muốn thêm budget hãy dùng warm-start mới.